# Movie Sentiment Analysis - Model Training

This notebook trains and compares multiple models for sentiment analysis on Rotten Tomatoes critic reviews.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

nltk.download('stopwords')

In [ ]:
# Load Data
df = pd.read_csv('../data/rotten_tomatoes_critic_reviews.csv')
df = df[df['top_critic'] == True].copy()
df = df.dropna(subset=['review_content', 'review_type'])
print(f"Loaded {len(df):,} reviews")

In [ ]:
# Preprocessing
base_stopwords = set(stopwords.words('english'))
custom_stopwords = [w for w in base_stopwords if w not in ['not', 'never', 'no', 'nor', 'none', 'neither']]

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in custom_stopwords])
    return text

df['clean_review'] = df['review_content'].apply(clean_text)
df['sentiment_label'] = df['review_type'].map({'Fresh': 1, 'Rotten': 0})

In [ ]:
# Split Data
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment_label'],
    test_size=0.2, random_state=42, stratify=df['sentiment_label']
)

In [ ]:
# Vectorize
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    stop_words=custom_stopwords,
    sublinear_tf=True
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Train Logistic Regression
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_vec, y_train)

# Evaluate
y_pred = model.predict(X_test_vec)
y_proba = model.predict_proba(X_test_vec)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=['Rotten', 'Fresh']))

In [ ]:
# Find Optimal Threshold
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
optimal_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal Threshold: {optimal_threshold:.3f}")

In [ ]:
# Save Model Pipeline
pipeline = {
    'LR': model,
    'vectorizer': vectorizer,
    'custom_stopwords': custom_stopwords,
    'threshold': optimal_threshold
}

joblib.dump(pipeline, '../backend/traditional_models.pkl')
print("✅ Model saved to backend/traditional_models.pkl")